> 在线讲义：[第 6 章 · attention 的起源：Bahdanau 的加性打分 vs 点积打分](https://wikieden.github.io/beyond-transformer/lessons/06-bahdanau.html) · 本书《超越 Transformer》

# 第 6 章 · attention 的起源：Bahdanau 的加性打分 vs 点积打分

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/06-bahdanau.html) · 仓库里 `beyond-transformer/notes/lessons/06-bahdanau.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 瓶颈 | d / n | `bottleneck()` | 0.80、0.20、0.05 |
| 第二步 加性打分 | vᵀtanh(Ws + Uh) | `additive_scores()`、`context()` | α = [0.4549, 0.1735, 0.3716] |
| 第三步 点积打分 | sᵀh | `dot_scores()` | α = [0.0634, 0.4683, 0.4683] |
| 第四步 参数量 | d_a(d_s + d_h + 1) | `param_count(1000, 2000, 1000)` | 3,001,000 |

e = vᵀ tanh(W s + U h) 和点积打分 e = sᵀh；softmax 成权重 α，加权求和得到上下文 c。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
H = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])   # 原文 3 个词的注释向量 h_1..h_3（编码器输出）
S = np.array([0.0, 2.0])                               # 解码器上一步的状态 s_{i−1}
W = np.array([[1.0, 0.0], [0.0, 1.0]])
U = np.array([[1.0, 0.0], [0.0, -1.0]])
V = np.array([1.0, 1.0])

**函数 `softmax`**：小工具。

In [ ]:
def softmax(e):
    z = np.exp(e - e.max())
    return z / z.sum()

**函数 `additive_scores`**：Bahdanau 附录 A.1.2（L813）：e_j = vᵀ tanh(W s + U h_j)；α 是式 (6) 的 softmax。

In [ ]:
def additive_scores(s=S, h=H, w=W, u=U, v=V):
    """Bahdanau 附录 A.1.2（L813）：e_j = vᵀ tanh(W s + U h_j)；α 是式 (6) 的 softmax。"""
    return np.array([v @ np.tanh(w @ s + u @ hj) for hj in h])

**函数 `dot_scores`**：小工具。

In [ ]:
def dot_scores(s=S, h=H):
    return h @ s

**函数 `context`**：小工具。

In [ ]:
def context(scores, h=H):
    a = softmax(scores)
    return a, a @ h

**函数 `param_count`**：加性打分要学 W（d_a × d_s）、U（d_a × d_h）、v（d_a）；点积打分什么都不用学（要求 d_s = d_h）。

In [ ]:
def param_count(d_s, d_h, d_a):
    """加性打分要学 W（d_a × d_s）、U（d_a × d_h）、v（d_a）；点积打分什么都不用学（要求 d_s = d_h）。"""
    return d_a * d_s + d_a * d_h + d_a

**函数 `bottleneck`**：固定长度向量：不管句子多长都只有 d 个数，平均每个词分到 d/n 个数。

In [ ]:
def bottleneck(lengths=(5, 20, 80), d=4):
    """固定长度向量：不管句子多长都只有 d 个数，平均每个词分到 d/n 个数。"""
    return {n: d / n for n in lengths}

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 瓶颈：整句压成一个 4 维向量")
for n, r in bottleneck().items():
    print(f"  句长 {n:2d}：每个词平均只分到 {r:.2f} 个数")
print("\n第二步 加性打分")
e = additive_scores()
a, c = context(e)
print(f"  W s = {(W @ S).tolist()}；e = {np.round(e, 4).tolist()}；α = {np.round(a, 4).tolist()}；c = {np.round(c, 4).tolist()}")

In [ ]:
print("\n第三步 点积打分")
e2 = dot_scores()
a2, c2 = context(e2)
print(f"  e = sᵀh = {e2.tolist()}；α = {np.round(a2, 4).tolist()}；c = {np.round(c2, 4).tolist()}")

In [ ]:
print("\n第四步 参数量与计算方式")
print(f"  d_s = d_h = 1000、d_a = 1000（原文设定）：加性打分要学 {param_count(1000, 2000, 1000):,} 个参数（双向 RNN 的 h 是 2000 维）")
print("  点积打分：0 个参数；n 个 query × m 个 key 一次矩阵乘法就算完")

## 3. 代码里才看得到的

- **加性打分是一个循环**：每个 h_j 都要单独过一遍 tanh 网络；点积是一行 `h @ s`。这就是“实践中快得多”的直观来源。
- **W s 可以先算好**：同一步里 W s 对所有 j 都一样，只算一次；Bahdanau 的附录里也把 U h_j 提前算好存起来。

## 4. 自己改一改

- 把 U 改成单位阵，加性打分的权重变成什么样？和点积的排序一致吗？
- 把 s 改成 [2, 0]，两种打分各自最看重哪个词？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 整句压一张卡片，句子一长就写不下。
> 每写一个词回头看，打分、softmax、加权求上下文。
> 加性打分：各乘一矩阵，加起来过 tanh，再用 v 收；
> 点积打分：方向一致就高分，没有参数，一次矩阵乘。
> 点积快又省，除以根号 d，就把加性比下去。